# Introduction to Machine Learning — Lecture 1

Unit 2 · ML Fundamentals & Preprocessing

Files needed: `Titanic-Dataset.csv`, `tips.csv`

In [71]:
! pip3 install scikit-learn

In [72]:
import numpy as np
import pandas as pd
import sklearn

In [73]:
from sklearn.neighbors import KNeighborsClassifier

---
## 1. What is Machine Learning

> **Flow:** Rules written by hand → rules learned from data.

### Practice 1

**Q1.** Gmail decides which mails go to the Spam folder.

- Version A: a rule is written — any mail containing the word "lottery" goes to Spam.
- Version B: millions of users have clicked "Report spam" over the years, and Gmail works
  out for itself what spam looks like.

Which is traditional programming, which is machine learning?

<details><summary>Answer</summary>

A is traditional programming — a person wrote the condition.
B is machine learning — the mails and the past answers are given, and the rule is learned.
</details>

**Q2.** Your phone unlocks when it sees your face. Why would it be very hard to write that
as a set of conditions by hand?

<details><summary>Answer</summary>

A face looks different with light, angle, glasses, a haircut, a beard. There is no small set
of conditions covering all of it — too many cases to write by hand, which is when ML is useful.
</details>

---
## 2. Supervised vs Unsupervised

> **Flow:** Does the data have an answer column, or not?

Today is supervised.

### Practice 2

**Q1.** Swiggy has records of every order it has delivered, and each record says whether that
delivery was late or on time. They want to predict this for a new order.
Supervised or unsupervised?

<details><summary>Answer</summary>

Supervised — the answer column already exists in the records.
</details>

**Q2.** Spotify has the listening history of every user. Nobody has ever put users into
categories. The company wants to find out whether natural groups of listeners exist.
Supervised or unsupervised?

<details><summary>Answer</summary>

Unsupervised — there is no answer column, so the goal is to find groups, not predict a
known answer.
</details>

---
## 3. Classification vs Regression

> **Flow:** Look at the target column, and only the target column.

Number in a range → regression. One of a fixed set → classification.

In [74]:
df = pd.read_csv('Titanic-Dataset.csv')
df['Survived'].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

### Practice 3

Classification or regression?

**Q1.** Ola shows you an estimated fare of Rs 247 before you book the ride.

**Q2.** GPay flags a transaction as suspicious or not suspicious.

**Q3.** Gmail sorts an incoming mail into one of three tabs — Primary, Social, or Promotions.

<details><summary>Answers</summary>

1. Regression — the fare is a number in a range.
2. Classification — two labels.
3. Classification — three labels. Numbering the tabs 1, 2, 3 would not make it regression.
</details>

---
## 4. The ML Pipeline

> **Flow:** Six stages. Every lecture in this course sits inside one of them.

`1 Clean → 2 Split → 3 Feature Engineering → 4 Train → 5 Predict → 6 Evaluate`

Stages 1–3 are most of the work.

---
## 5. scikit-learn

> **Flow:** One tool per stage. Two patterns.

Preprocessor: `.fit()` → `.transform()`
Model: `.fit()` → `.predict()`


## 6. Stage 1 — Data Cleaning

> **Flow:** Load, find what is broken, fix it.

**Question:** given a passenger's details, did they survive?

In [75]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [76]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Out of 891 rows: `Age` has 714, `Cabin` has 204, `Embarked` has 889.

In [77]:
# Drop unusable columns, fill Age, drop 2 rows



`Cabin` is 77% empty. `Name`, `Ticket`, `PassengerId` are unique per row, so no pattern.

In [78]:
df=df.drop(columns=['Name','Cabin','Ticket','PassengerId'])
df.dropna(subset=['Embarked'],inplace=True)

In [79]:
df['Age']=df['Age'].fillna(df['Age'].median())


In [80]:
df.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [81]:
x=df.drop(columns='Survived')
y=df['Survived']
print(x.shape)
print(y.shape)


(889, 7)
(889,)


---
## 8. Stage 2 — Train/Test Split

> **Flow:** Hide some rows before preparing anything.

In [82]:
from sklearn.model_selection import train_test_split

In [83]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

`random_state` — same rows for everyone. `stratify` — same survival ratio in both halves.

---
## 9. Stage 3 — Encoding

> **Flow:** Turn text columns into numbers.

Nominal → one-hot. Ordinal → numbered in order. Label → target only.

In [84]:
# Fit the encoder on training rows only

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)


In [85]:
x_train.info()

<class 'pandas.DataFrame'>
Index: 711 entries, 708 to 103
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    711 non-null    int64  
 1   Sex       711 non-null    str    
 2   Age       711 non-null    float64
 3   SibSp     711 non-null    int64  
 4   Parch     711 non-null    int64  
 5   Fare      711 non-null    float64
 6   Embarked  711 non-null    str    
dtypes: float64(2), int64(3), str(2)
memory usage: 44.4 KB


In [86]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
cat_cols=['Sex','Embarked']

In [89]:
ct=ColumnTransformer([('cat',OneHotEncoder(sparse_output=False),cat_cols)],
                     remainder='passthrough')
x_train_final=ct.fit_transform(x_train)
x_test_final=ct.transform(x_test)


In [90]:
print(x_train.shape)
print(x_train_final.shape)

(711, 7)
(711, 10)


In [91]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


In [95]:
model=KNeighborsClassifier()
model.fit(x_train_final,y_train)
y_pre=model.predict(x_test_final)
print('this the accuracy score',accuracy_score(y_test,y_pre))

this the accuracy score 0.6853932584269663


In [96]:
from sklearn.preprocessing import StandardScaler


In [97]:
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train_final)
x_test_scaled=scaler.fit_transform(x_test_final)

In [100]:
scaled_model=KNeighborsClassifier()
scaled_model.fit(x_train_scaled,y_train)
y_pre_scaled=scaled_model.predict(x_test_scaled)
print("before scaling",accuracy_score(y_test,y_pre))
print("After Scaling ",accuracy_score(y_test,y_pre_scaled))

before scaling 0.6853932584269663
After Scaling  0.7752808988764045
